# 5.3 Verification

This notebook implements the verification tests from thesis Chapter 5.3 (Table 5.1). Each cell checks one invariant. The three blocks are:

1. **Unit tests** for the core computational functions (Leontief, NPV, severance pay, wage curve, AI learning curve).
2. **Conservation checks** at model level (worker, task, and cost conservation).
3. **Edge cases** that test the model response under extreme parameters (cheap/expensive AI, no institutions, fully permanent workforce).

All tests produce a `PASS` line when the invariant has been verified. An `AssertionError` means that the implementation deviates from the specification in Chapter 4.

## Setup
Imports, path configuration, and a helper function for quickly creating a model with overridden parameters.

In [1]:
import os, sys
import numpy as np

# Add the model directory to sys.path so LabourMarketModel can be imported.
def find_repo_root(start):
    here = os.path.abspath(start)
    while True:
        model_dir = os.path.join(here, 'model')
        if os.path.exists(os.path.join(model_dir, 'labour_market_model.py')):
            return here
        parent = os.path.dirname(here)
        if parent == here:
            raise FileNotFoundError('Could not find repo root containing model/labour_market_model.py')
        here = parent

REPO_ROOT = find_repo_root(os.getcwd())
MODEL_DIR = os.path.join(REPO_ROOT, 'model')
if MODEL_DIR not in sys.path:
    sys.path.insert(0, MODEL_DIR)

from labour_market_model import LabourMarketModel, leontief_output, leontief_bottlenecks
from base_parameters import BASE_PARAMS

def make_model(**overrides):
    """Build a model from BASE_PARAMS and apply any overrides."""
    params = BASE_PARAMS.copy()
    params.update(overrides)
    # adoption_mode is not part of BASE_PARAMS; default = 'ulc'
    params.setdefault('adoption_mode', 'ulc')
    return LabourMarketModel(**params)

print('Setup OK - model directory:', MODEL_DIR)

Setup OK - model directory: c:\<repo>\model


---
## Block 1 - Unit Tests

### Test 1 - Leontief Output

**Invariant:** $y_j = \min_x \frac{u_x}{c_x}$.

We test three analytically solvable input-output pairs, plus the error paths (negative input, non-positive coefficient) and the empty-input convention.

In [2]:
# Symmetric case: all ratios equal -> output = ratio
assert leontief_output({1: 10, 2: 10, 3: 10}, {1: 1, 2: 1, 3: 1}) == 10.0

# Asymmetric: bottleneck on task 2
assert leontief_output({1: 100, 2: 5, 3: 80}, {1: 1, 2: 1, 3: 1}) == 5.0

# Non-unit coefficients: 10/2=5, 20/5=4 -> min = 4
assert leontief_output({1: 10, 2: 20}, {1: 2, 2: 5}) == 4.0

# Bottleneck detection: only task 2 should attain the minimum
U, bn = leontief_bottlenecks({1: 100, 2: 5, 3: 80}, {1: 1, 2: 1, 3: 1})
assert U == 5.0 and bn == [2]

# Empty input: convention is 0
assert leontief_output({}, {}) == 0.0

# Error paths: negative u or c <= 0 should raise ValueError
for bad_args in [({1: -1}, {1: 1}), ({1: 1}, {1: 0}), ({1: 1}, {1: -1})]:
    try:
        leontief_output(*bad_args)
    except ValueError:
        pass
    else:
        raise AssertionError(f'Expected ValueError for {bad_args}')

print('PASS - Test 1: Leontief output y = min_x(u_x/c_x) verified')

PASS - Test 1: Leontief output y = min_x(u_x/c_x) verified


### Test 2 - NPV Calculation (All Four Modes)

**Invariant:** 
$$\mathrm{NPV} = -(I + S) + \sum_{t=1}^{T} \frac{\Delta C_t}{(1+r)^t}$$

with $\Delta C_t = uc^h_t - uc^{ai}_t$. We analytically reproduce the result of `Producer.calculate_npv` for `npv_naive`, `npv_adaptive`, and `npv_mean_field`. For `ulc` mode, we confirm that the adoption decision is a direct unit-cost comparison (no NPV).

In [3]:
def analytical_npv(prod, task, m):
    """Reproduce the NPV formula independently of calculate_npv()."""
    T = m.T_horizon
    ts = np.arange(1, T + 1)
    prod_h = max(prod.productivity_human(task, 'high'), 1e-9)
    prod_l = max(prod.productivity_human(task, 'low'), 1e-9)
    prod_ai = max(prod.productivity_ai(task), 1e-9)
    future_k = m.k_ai_floor + (m.k_ai_0 - m.k_ai_floor) * np.exp(-m.k_ai_decay * (m.current_step + ts))
    ai_uc = future_k / prod_ai
    if m.adoption_mode == 'npv_naive':
        human_uc = np.full(T, min(m.w_h / prod_h, m.w_l / prod_l))
    elif m.adoption_mode == 'npv_adaptive':
        floor_h = max(m.w_min, m.a_h)
        floor_l = max(m.w_min, m.a_l)
        exp_w_h = np.maximum(floor_h, m.w_h + ts * m.dw_h_smoothed)
        exp_w_l = np.maximum(floor_l, m.w_l + ts * m.dw_l_smoothed)
        human_uc = np.minimum(exp_w_h / prod_h, exp_w_l / prod_l)
    elif m.adoption_mode == 'npv_mean_field':
        Lh = sum(1 for w in m.workers if w.employed and w.skill_level == 'high')
        Ll = sum(1 for w in m.workers if w.employed and w.skill_level == 'low')
        total = Lh + Ll
        share_h = Lh / total if total > 0 else 0.5
        Lh_proj = np.maximum(0.0, Lh - ts * m.displacement_flow_smoothed * share_h)
        Ll_proj = np.maximum(0.0, Ll - ts * m.displacement_flow_smoothed * (1 - share_h))
        exp_w_h = np.maximum(m.w_min, m.a_h + m.b_h * Lh_proj)
        exp_w_l = np.maximum(m.w_min, m.a_l + m.b_l * Ll_proj)
        human_uc = np.minimum(exp_w_h / prod_h, exp_w_l / prod_l)
    discount = (1 + m.r_discount) ** ts
    severance = prod.severance_cost_for_task(task)
    return -(task.investment_cost + severance) + float(np.sum((human_uc - ai_uc) / discount))

for mode in ['npv_naive', 'npv_adaptive', 'npv_mean_field']:
    m = make_model(adoption_mode=mode, employment_protection=False)
    prod = m.producers[0]
    task = prod.tasks[0]
    expected = analytical_npv(prod, task, m)
    actual = prod.calculate_npv(task)
    assert abs(expected - actual) < 1e-9, f'{mode}: {expected} vs {actual}'
    print(f'  {mode:18s}  NPV_analytical={expected:+.4f}  NPV_model={actual:+.4f}  ok')

# ULC mode: should_automate = (AI unit cost < cheapest human unit cost)
m = make_model(adoption_mode='ulc')
prod = m.producers[0]
task = prod.tasks[0]
uc_ai = prod.unit_cost_ai(task, m.k_ai)
uc_h = prod.unit_cost_human(task, m.w_h, 'high')
uc_l = prod.unit_cost_human(task, m.w_l, 'low')
ulc_decision, npv = prod.should_automate(task)
assert npv is None
assert ulc_decision == (uc_ai < min(uc_h, uc_l))
print(f'  ulc                direct comparison uc_ai vs min(uc_h, uc_l)  ok')

print('PASS - Test 2: NPV reproducible for all four adoption modes')

  npv_naive           NPV_analytical=+72.7306  NPV_model=+72.7306  ok
  npv_adaptive        NPV_analytical=+72.7306  NPV_model=+72.7306  ok
  npv_mean_field      NPV_analytical=+72.7306  NPV_model=+72.7306  ok
  ulc                direct comparison uc_ai vs min(uc_h, uc_l)  ok
PASS - Test 2: NPV reproducible for all four adoption modes


### Test 3 - Severance Pay

**Invariant:** For each permanent worker,
$$S_w = \text{severance\_rate} \cdot w \cdot \frac{\text{tenure}}{\text{steps\_per\_year}}$$

and task-level severance is the sum over permanent workers only. Flexible workers generate no severance, and when `employment_protection = False`, severance is identically zero.

In [4]:
# Case A - protection on: the per-worker formula should hold
m = make_model(adoption_mode='ulc', employment_protection=True)
checked = 0
for prod in m.producers:
    for task in prod.tasks:
        expected = sum(
            m.severance_rate * w.wage * (w.tenure / m.steps_per_year)
            for w in task.employees if w.contract_type == 'vast'
        )
        actual = prod.severance_cost_for_task(task)
        assert abs(expected - actual) < 1e-9, f'{expected} vs {actual}'
        checked += 1
print(f'  {checked} tasks checked; formula (1/3)*w*tenure-years reproduced exactly')

# Case B - protection off: all severance = 0
m_off = make_model(adoption_mode='ulc', employment_protection=False)
for prod in m_off.producers:
    for task in prod.tasks:
        assert prod.severance_cost_for_task(task) == 0.0
print('  employment_protection=False => severance == 0 for all tasks  ok')

# Case C - flexible workers generate no severance
m = make_model(adoption_mode='ulc', employment_protection=True, init_share_vast=0.0)
totals = [prod.severance_cost_for_task(t) for prod in m.producers for t in prod.tasks]
assert all(s == 0.0 for s in totals), 'flexible workers must not generate severance'
print('  init_share_vast=0  =>  severance = 0 (flexible workers generate no severance pay)  ok')

print('PASS - Test 3: Severance pay satisfies the WAB formula')

  400 tasks checked; formula (1/3)*w*tenure-years reproduced exactly
  employment_protection=False => severance == 0 for all tasks  ok
  init_share_vast=0  =>  severance = 0 (flexible workers generate no severance pay)  ok
PASS - Test 3: Severance pay satisfies the WAB formula


### Test 4 - Wage Curve and Partial Adjustment (Half-Life)

**Invariant:** Under partial adjustment $w_{t+1} = \lambda \cdot w^{*} + (1-\lambda)\cdot w_t$, the deviation decays as $(1-\lambda)^t$. The half-life after a shock is
$$t_{1/2} = \frac{\ln 2}{-\ln(1-\lambda)}.$$

We test that (a) the discrete recursion crosses the analytical half-life point at $\lceil t_{1/2}\rceil$, and (b) `LabourMarketModel.update_wages` behaves identically after a shock to `w_h` (with employment held fixed).

In [5]:
# (a) Pure recursion
for lam in [0.10, 0.25, 0.5, 0.8]:
    target = 5.0
    w = 1.0
    dev0 = abs(w - target)
    expected = np.log(2) / -np.log(1 - lam)
    crossing = None
    for t in range(1, 1000):
        w = lam * target + (1 - lam) * w
        if abs(w - target) <= dev0 / 2:
            crossing = t
            break
    assert crossing == int(np.ceil(expected)), f'lam={lam}: {crossing} vs ceil({expected:.3f})'
    print(f'  lambda={lam:4.2f}  theory t_1/2 = {expected:6.3f}  recursion crosses at step {crossing}  ok')

# (b) Full model: hold employment fixed and shock w_h
m = make_model(adoption_mode='ulc', lam=0.25)
Lh = sum(1 for w in m.workers if w.employed and w.skill_level == 'high')
target_h = max(m.w_min, m.a_h + m.b_h * Lh)
m.w_h = target_h * 0.5  # 50% downward shock
dev0 = abs(m.w_h - target_h)
expected = np.log(2) / -np.log(1 - m.lam)
crossing = None
for t in range(1, 200):
    m.update_wages()  # no step() => Lh remains constant
    if abs(m.w_h - target_h) <= dev0 / 2 and crossing is None:
        crossing = t
        break
assert crossing == int(np.ceil(expected)), f'model: {crossing} vs {expected}'
print(f'  model: after shock, w_h crosses the half-life point at step {crossing} (theory {expected:.3f})  ok')

print('PASS - Test 4: t_1/2 = ln(2)/-ln(1-lambda) holds for the wage curve')

  lambda=0.10  theory t_1/2 =  6.579  recursion crosses at step 7  ok
  lambda=0.25  theory t_1/2 =  2.409  recursion crosses at step 3  ok
  lambda=0.50  theory t_1/2 =  1.000  recursion crosses at step 1  ok
  lambda=0.80  theory t_1/2 =  0.431  recursion crosses at step 1  ok
  model: after shock, w_h crosses the half-life point at step 3 (theory 2.409)  ok
PASS - Test 4: t_1/2 = ln(2)/-ln(1-lambda) holds for the wage curve


### Test 5 - AI Learning Curve (Exponential Decay)

**Invariant:** $k_{ai}(t) = k_{ai,\text{floor}} + (k_{ai,0}-k_{ai,\text{floor}})\cdot e^{-\delta t}$.

We test (a) that the model value of `k_ai` after an arbitrary number of steps exactly satisfies this formula, and (b) that the series converges to `k_ai_floor` over sufficiently long runs.

In [6]:
m = make_model(adoption_mode='npv_naive')
k_floor, k0, decay = m.k_ai_floor, m.k_ai_0, m.k_ai_decay

# (a) Formula reproduced after an arbitrary number of steps
for n in [1, 5, 25, 100]:
    m_test = make_model(adoption_mode='npv_naive')
    for _ in range(n):
        m_test.step()
    expected = k_floor + (k0 - k_floor) * np.exp(-decay * n)
    assert abs(m_test.k_ai - expected) < 1e-9, f'n={n}: {m_test.k_ai} vs {expected}'
    print(f'  after {n:3d} steps: model k_ai={m_test.k_ai:8.5f}  formula={expected:8.5f}  ok')

# (b) Convergence: after 800 steps, k_ai should be very close to the floor
m_long = make_model(adoption_mode='npv_naive')
for _ in range(800):
    m_long.step()
distance = m_long.k_ai - k_floor
assert distance >= 0 and distance < 1e-3, f'k_ai - floor = {distance}'
print(f'  t=800: k_ai - floor = {distance:.2e}  -> convergence to k_ai_floor  ok')

print('PASS - Test 5: AI learning curve converges to k_ai_floor at rate delta')

  after   1 steps: model k_ai=39.60397  formula=39.60397  ok


  after   5 steps: model k_ai=38.09675  formula=38.09675  ok
  after  25 steps: model k_ai=32.13061  formula=32.13061  ok
  after 100 steps: model k_ai=22.70671  formula=22.70671  ok
  t=800: k_ai - floor = 2.25e-06  -> convergence to k_ai_floor  ok
PASS - Test 5: AI learning curve converges to k_ai_floor at rate delta


---
## Block 2 - Conservation Checks

### Test 6 - Worker Conservation

**Invariant:** $|\text{employed}| + |\text{unemployed}| = N$ throughout the run, and the composition by skill level remains constant (no workers are created or destroyed).

In [7]:
m = make_model(adoption_mode='npv_mean_field')
N = len(m.workers)
N_high = sum(1 for w in m.workers if w.skill_level == 'high')
N_low  = sum(1 for w in m.workers if w.skill_level == 'low')

for t in range(1, 151):
    m.step()
    n_emp   = sum(1 for w in m.workers if w.employed)
    n_unemp = sum(1 for w in m.workers if not w.employed)
    assert n_emp + n_unemp == N, f'step {t}: {n_emp}+{n_unemp} != {N}'
    nh = sum(1 for w in m.workers if w.skill_level == 'high')
    nl = sum(1 for w in m.workers if w.skill_level == 'low')
    assert nh == N_high and nl == N_low, f'step {t}: skill distribution shifted'

print(f'PASS - Test 6: after 150 steps, N={N} (high={N_high}, low={N_low}) remains constant')

PASS - Test 6: after 150 steps, N=450 (high=200, low=250) remains constant


### Test 7 - Task Conservation

**Invariant:** The number of tasks per producer (`n_tasks`, default 20) is a structural constant; automation changes only the assignment (`automated` flag), not the cardinality of `producer.tasks`.

In [8]:
expected_n_tasks = 20
m = make_model(adoption_mode='ulc')

# Initial state
for prod in m.producers:
    assert len(prod.tasks) == expected_n_tasks

# Throughout the run
for t in range(1, 151):
    m.step()
    for prod in m.producers:
        assert len(prod.tasks) == expected_n_tasks, f'producer {prod.unique_id} step {t}: {len(prod.tasks)} tasks'
        # task_ids remain unique
        ids = [task.task_id for task in prod.tasks]
        assert len(set(ids)) == expected_n_tasks

print(f'PASS - Test 7: |tasks| = {expected_n_tasks} per producer over 150 steps')

PASS - Test 7: |tasks| = 20 per producer over 150 steps


### Test 8 - Cost Identity

**Invariant:** $\text{total variable costs}_j = \sum w_i + \sum (n_{ai} \cdot k_{ai})$ per firm, and also in aggregate at model level.

In [9]:
m = make_model(adoption_mode='npv_mean_field')
for _ in range(1000):
    m.step()

# Per firm
for prod in m.producers:
    expected = prod.labour_costs + prod.ai_costs
    assert abs(prod.total_costs - expected) < 1e-9, f'firm {prod.unique_id}: {prod.total_costs} vs {expected}'

# Aggregate
total_wages = sum(prod.labour_costs for prod in m.producers)
total_ai    = sum(prod.ai_costs    for prod in m.producers)
total_costs = sum(prod.total_costs for prod in m.producers)
assert abs(total_costs - (total_wages + total_ai)) < 1e-9

# Cross-check against worker.wage (only workers on non-automated tasks count)
sum_worker_wages = sum(
    w.wage for prod in m.producers for t in prod.tasks if not t.automated for w in t.employees
)
assert abs(total_wages - sum_worker_wages) < 1e-6

print(f'  sum wages       = {total_wages:12.2f}')
print(f'  sum AI rental   = {total_ai:12.2f}')
print(f'  sum total VC    = {total_costs:12.2f}  =  {total_wages + total_ai:12.2f}  ok')
print('PASS - Test 8: total variable costs = sum wages + sum AI rental')

  sum wages       =      5964.45
  sum AI rental   =      3300.00
  sum total VC    =      9264.45  =       9264.45  ok
PASS - Test 8: total variable costs = sum wages + sum AI rental


---
## Block 3 - Edge Cases

### Test 9 - Trivially Cheap AI => Adoption to 100%

**Invariant:** With `k_ai_floor -> 0` (and `k_ai = k_ai_floor`), AI becomes cheaper than human labour for every task; in all adoption modes, the automation share converges to 1. We set `p_evaluate=1.0` so no task remains unevaluated.

In [10]:
m = make_model(
    adoption_mode='npv_mean_field',
    k_ai=0.001,
    k_ai_floor=0.001,
    p_evaluate=1.0,
    severance_rate=0.01,
    I_base = 0.0
)
for _ in range(960):
    m.step()

n_total = sum(len(p.tasks) for p in m.producers)
n_auto  = sum(p.n_automated for p in m.producers)
share   = n_auto / n_total
print(f'  k_ai_floor=0.001  =>  automated {n_auto}/{n_total}  ({share:.1%})')
assert share == 1.0, f'expected 100% adoption, got {share:.1%}'
print('PASS - Test 9: cheap AI => full adoption')

  k_ai_floor=0.001  =>  automated 400/400  (100.0%)
PASS - Test 9: cheap AI => full adoption


### Test 10 - Prohibitively Expensive AI => Adoption to 0%

**Invariant:** When `k_ai = k_ai_floor -> infinity`, the AI unit cost is always higher than the human unit cost and the NPV remains negative. No task is automated.

In [11]:
m = make_model(
    adoption_mode='npv_naive',
    k_ai=10_000.0,
    k_ai_floor=10_000.0,
    p_evaluate=1.0,
)
for _ in range(150):
    m.step()

n_auto = sum(p.n_automated for p in m.producers)
print(f'  k_ai_floor=10 000  =>  automated tasks: {n_auto}')
assert n_auto == 0, f'expected 0 automation, got {n_auto}'
print('PASS - Test 10: prohibitively expensive AI => 0% adoption')

  k_ai_floor=10 000  =>  automated tasks: 0
PASS - Test 10: prohibitively expensive AI => 0% adoption


### Test 11 - `employment_protection = False` => Institutional Effect = 0

**Invariant:** Without employment protection, the WAB component disappears from the model: severance is identically zero, chain-rule conversions are identically zero, and non-renewals are identically zero. A high `chain_limit` (effectively infinity) confirms that the institutional effect is inactive even without the protection flag.

In [12]:
m = make_model(
    adoption_mode='ulc',
    employment_protection=False,
    chain_limit=10**9,  # effectively infinite
)

total_severance_seen = 0.0
for _ in range(120):
    m.step()
    # Severance is zero on every task
    for prod in m.producers:
        for task in prod.tasks:
            assert prod.severance_cost_for_task(task) == 0.0
    # No chain-rule movements
    assert m._conversions_this_step == 0
    assert m._non_renewals_this_step == 0

# No permanent contracts generated by warm-start (employment_protection off => contracts remain default 'flex')
n_vast = sum(1 for w in m.workers if w.contract_type == 'vast')
assert n_vast == 0, f'expected 0 permanent contracts without protection, got {n_vast}'

print(f'  conversions: 0   non-renewals: 0   severance: 0   #permanent: {n_vast}  ok')
print('PASS - Test 11: without protection + chain_limit->infinity, the institutional effect vanishes')

  conversions: 0   non-renewals: 0   severance: 0   #permanent: 0  ok
PASS - Test 11: without protection + chain_limit->infinity, the institutional effect vanishes


### Test 12 - `init_share_vast = 1` => Maximum Severance Component

**Invariant:** With all initially employed workers on permanent contracts, the severance component for proactive automation reaches its maximum. We test this by comparing total severance at `t = 0` (after warm-start) with the variant where `init_share_vast = 0`.

In [13]:
def total_severance(m):
    return sum(prod.severance_cost_for_task(t) for prod in m.producers for t in prod.tasks)

m_max = make_model(adoption_mode='ulc', employment_protection=True, init_share_vast=1.0)
m_min = make_model(adoption_mode='ulc', employment_protection=True, init_share_vast=0.0)

S_max = total_severance(m_max)
S_min = total_severance(m_min)

print(f'  init_share_vast=1.0   ->  sum severance = {S_max:10.2f}')
print(f'  init_share_vast=0.0   ->  sum severance = {S_min:10.2f}')

assert S_min == 0.0, 'flexible workers must not generate severance'
assert S_max > 0.0, 'expected positive severance with a fully permanent workforce'

# All employed workers must contractually be permanent
employed_max = [w for w in m_max.workers if w.employed]
n_vast = sum(1 for w in employed_max if w.contract_type == 'vast')
assert n_vast == len(employed_max), f'{n_vast}/{len(employed_max)} permanent'
print(f'  all {n_vast}/{len(employed_max)} employed workers are permanent  ok')

# And the maximum claim: no alternative init_share_vast in (0,1) may produce higher severance
for share in [0.25, 0.5, 0.75]:
    m_x = make_model(adoption_mode='ulc', employment_protection=True, init_share_vast=share)
    S_x = total_severance(m_x)
    print(f'  init_share_vast={share:>4.2f}  ->  sum severance = {S_x:10.2f}')
    assert S_x <= S_max + 1e-6, f'init_share_vast={share}: {S_x} > S_max={S_max}'

print('PASS - Test 12: init_share_vast=1 yields the maximum severance component')

  init_share_vast=1.0   ->  sum severance =   22840.54
  init_share_vast=0.0   ->  sum severance =       0.00
  all 400/400 employed workers are permanent  ok
  init_share_vast=0.25  ->  sum severance =    5425.20
  init_share_vast=0.50  ->  sum severance =   11214.37
  init_share_vast=0.75  ->  sum severance =   17195.67
PASS - Test 12: init_share_vast=1 yields the maximum severance component
